# 실습 문제 모범답안 - 07. Guardrails

`07_guardrails` 노트북의 실습 문제에 대한 모범답안 예시입니다.

### 문제 요약

1. **입력 가드레일**:
   사용자 입력에 **금지어(`해킹`, `폭탄`)** 가 포함되어 있으면 트립와이어를 발동시켜
   에이전트 실행을 **중단**시키세요. (힌트: `@input_guardrail`)

2. **출력 가드레일**:
   에이전트의 응답에 **URL(링크)** 이 포함되어 있으면 트립와이어를 발동시켜
   차단하세요. (힌트: `@output_guardrail` + 정규식)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
Model = "gpt-5.4-mini"

## 문제 1. 입력 가드레일 - 금지어 차단

- 금지어 검사는 단순 문자열 포함 여부이므로 **LLM 없이** 코드만으로 판단합니다.
  (가드레일이 반드시 LLM을 사용할 필요는 없습니다 - 빠르고 비용이 들지 않는 방식이 우선)
- `input_data`는 문자열 또는 메시지 리스트일 수 있으므로 두 경우 모두 처리합니다.

In [3]:
from agents import (Agent, GuardrailFunctionOutput, Runner,
    RunContextWrapper, TResponseInputItem, input_guardrail,
    InputGuardrailTripwireTriggered)

BANNED_WORDS = ["해킹", "폭탄"]

@input_guardrail
async def banned_word_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_data: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    """사용자 입력에 금지어가 포함되어 있으면 트립와이어를 발동합니다."""

    # 문자열이면 그대로, 메시지 리스트이면 content들을 이어붙여 검사 대상 텍스트 생성
    if isinstance(input_data, str):
        text = input_data
    else:
        text = " ".join(str(item.get("content", "")) for item in input_data)

    # 입력에 포함된 금지어 목록 추출
    found = [word for word in BANNED_WORDS if word in text]

    return GuardrailFunctionOutput(
        output_info={"found_banned_words": found},  # 어떤 금지어가 걸렸는지 기록
        tripwire_triggered=len(found) > 0,          # 금지어가 하나라도 있으면 차단
    )

## 문제 2. 출력 가드레일 - URL 차단

- 정규식으로 응답에 URL(`http://`, `https://`, `www.` 시작)이 있는지 검사합니다.

In [4]:
import re
from agents import output_guardrail, OutputGuardrailTripwireTriggered

# http(s):// 또는 www. 로 시작하는 URL 패턴
URL_PATTERN = r'(https?://\S+|www\.\S+)'

@output_guardrail
async def url_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str
) -> GuardrailFunctionOutput:
    """에이전트 응답에 URL이 포함되어 있으면 트립와이어를 발동합니다."""

    urls = re.findall(URL_PATTERN, str(output))

    return GuardrailFunctionOutput(
        output_info={"found_urls": urls},      # 감지된 URL 목록 기록
        tripwire_triggered=len(urls) > 0,      # URL이 하나라도 있으면 차단
    )

## 두 가드레일을 등록한 에이전트

In [5]:
assistant = Agent(
    name="안전한 도우미",
    instructions="""당신은 친절한 도우미입니다.
사용자가 참고할 사이트나 링크를 요청하면 실제 URL 주소를 포함해서 알려주세요.""",
    model=Model,
    input_guardrails=[banned_word_guardrail],  # 실행 전: 금지어 검사
    output_guardrails=[url_guardrail],         # 실행 후: URL 검사
)

## 테스트

In [8]:
# 테스트용 공통 함수: 입력/출력 가드레일 예외를 구분해서 출력
async def ask(prompt):
    print(f"질문: {prompt}")
    try:
        result = await Runner.run(assistant, prompt)
        print(f"✅ 응답: {result.final_output}")
    except InputGuardrailTripwireTriggered as e:
        print(f"🚫 입력 가드레일 발동! → 실행 중단")
        print(f"   상세: {e.guardrail_result.output.output_info}")
    except OutputGuardrailTripwireTriggered as e:
        print(f"🚫 출력 가드레일 발동! → 응답 차단")
        print(f"   상세: {e.guardrail_result.output.output_info}")
    print("-" * 100)

# 1) 정상 질문 → 두 가드레일 모두 통과
await ask("영어로 사랑이 뭐야?")

# 2) 금지어 포함 질문 → 입력 가드레일 트리거 → 실행 중단 
await ask("어제 뉴스에 나온 해킹 사고에 대해 말해줘")

# 3) URL 응답 유도 질문 → 출력에 URL 포함 시 출력 가드레일 트리거 → 차단
await ask("미국 국무부 홈페이지 URL을 알려줘")

질문: 영어로 사랑이 뭐야?
✅ 응답: “사랑” is **love** in English.
----------------------------------------------------------------------------------------------------
질문: 어제 뉴스에 나온 해킹 사고에 대해 말해줘
🚫 입력 가드레일 발동! → 실행 중단
   상세: {'found_banned_words': ['해킹']}
----------------------------------------------------------------------------------------------------
질문: 미국 국무부 홈페이지 URL을 알려줘
🚫 출력 가드레일 발동! → 응답 차단
   상세: {'found_urls': ['https://www.state.gov/']}
----------------------------------------------------------------------------------------------------


### 정리

| 문제 | 가드레일 | 검사 방식 | 트립와이어 조건 |
|------|---------|----------|---------------|
| 1. 금지어 차단 | `@input_guardrail` | 문자열 포함 여부 (LLM 불필요) | 입력에 `해킹`/`폭탄` 포함 |
| 2. URL 차단 | `@output_guardrail` | 정규식 `re.findall` | 응답에 `http(s)://` 또는 `www.` URL 포함 |

- **입력 가드레일**은 에이전트 실행 **전**에 차단하므로 고비용 모델 호출 자체가 발생하지 않습니다.
- **출력 가드레일**은 응답 생성 **후** 검사하므로, 모델이 URL을 만들었더라도 사용자에게 전달되기 전에 차단합니다.
- 가드레일 검사 로직은 LLM(예: 본문의 `guardrail_agent`)뿐 아니라 이 답안처럼 **단순 코드(문자열/정규식)** 로도 구현할 수 있습니다.